
# 練習問題: ガチャ (クーポンコレクター問題)

## 目標

「N 種類の景品が等確率で出るガチャを, **全種類そろうまで**引くと, 平均何回引くことになるか?」をモンテカルロ法で推定する。多数の独立な試行を `parallel for` で分担し, 結果を `reduction` で集計する, という並列計算の典型パターンを身につける。

## 課題

`gacha.cpp` (または `gacha.f90`) は, 1回の試行 (全種そろうまで引く) を行う関数 `one_trial` を持ち, それを `T` 回繰り返して引き回数の平均と標準偏差を求める。
各試行は**互いに独立** (それぞれ別の乱数列を使う) なので, 並列化できる。

`TODO` の箇所に **OpenMP の指示を追加** し, 試行ループを並列化して平均 (`sum`) と二乗和 (`sumsq`) を集計せよ。

- C++: ループの直前に `#pragma omp parallel for reduction(+:sum,sumsq)` を1行加える。
- Fortran: `do` ループを `!$omp parallel do reduction(+:sum,sumsq)` と `!$omp end parallel do` で囲む。

`reduction(+:sum,sumsq)` のように, 複数の変数を同時に縮約できる。各試行の乱数は試行番号から作るので, スレッドが分担しても結果は壊れない。

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore gacha.cpp -o gacha.exe

# Fortran
nvfortran -fast -mp=multicore gacha.f90 -o gacha.exe
```

第1引数で景品の種類数 `N`, 第2引数で試行回数 `T` を変えられる。

```
OMP_NUM_THREADS=4 ./gacha.exe 10 1000000
```

## 期待される結果

クーポンコレクター問題の理論では, 全種そろえるのに必要な平均回数は `N × H_N` (H_N = 1 + 1/2 + … + 1/N)。

- N=10 → 約 29.3 回
- N=50 → 約 224.9 回

```
N=10, trials=1000000: 平均 29.29 回 (理論 29.29), 標準偏差 11.2
```

- 試行回数 `T` を増やすほど, 推定した平均が理論値に近づく。
- **標準偏差が大きい** (上振れ・下振れが激しい) ことにも注目: 運が悪いと理論平均の倍以上引くこともある, というガチャの「沼」が数字で見える。
- `N` を変えて, 種類が増えると一気に大変になる (N に対しておよそ N·log N で増える) ことを確かめよ。

参考: スレッド数を増やすと (試行が独立なので) ほぼ比例して速くなる。台数効果も観察できる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit_a` (Aquarius) などでジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ gacha.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>

/* 試行ごとに独立な乱数生成器 (MINSTD 法)。
   16807*s は 64bit に収まるので桁あふれしない。返り値は [0,N) の整数。 */
static inline int draw(long long * s, int N) {
  *s = (16807LL * *s) % 2147483647LL;
  return (int)(*s % N);
}

/* 1試行: N 種類の景品が等確率で出るとき, 全種類そろうまでに引いた回数。
   collected を 64bit のビットマスクで管理する (N <= 62 を想定)。 */
long one_trial(int N, long long seed) {
  long long s = seed % 2147483647LL;
  if (s <= 0) s += 2147483646LL;
  unsigned long long got = 0;
  unsigned long long full = (N == 64 ? ~0ULL : ((1ULL << N) - 1));
  long draws = 0;
  while (got != full) {
    int k = draw(&s, N);
    got |= (1ULL << k);
    draws++;
  }
  return draws;
}

int main(int argc, char ** argv) {
  int  N = (argc > 1 ? atoi(argv[1]) : 10);          /* 景品の種類数 */
  long T = (argc > 2 ? atol(argv[2]) : 1000000);     /* 試行回数 */
  double sum = 0.0, sumsq = 0.0;

  /* T 回の試行は互いに独立。各試行の引き回数 d を集計する。 */
  // TODO: 各試行は独立なので #pragma omp parallel for reduction(+:sum,sumsq) で並列化・集計せよ.
  for (long t = 0; t < T; t++) {
    long d = one_trial(N, (long long)(t + 1) * 2654435761LL + 12345LL);
    sum   += (double)d;
    sumsq += (double)d * (double)d;
  }

  double mean = sum / T;
  double var  = sumsq / T - mean * mean;
  /* 理論期待値 = N * H_N (H_N は調和数) */
  double H = 0.0;
  for (int k = 1; k <= N; k++) H += 1.0 / k;
  printf("N=%d, trials=%ld: 平均 %.3f 回 (理論 %.3f), 標準偏差 %.3f\n",
         N, T, mean, N * H, sqrt(var));
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore gacha.cpp -o gacha_cpp.exe

## 2-2. 実行
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./gacha_cpp.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a

./gacha_cpp.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gacha_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ gacha.f90
module gacha_mod
contains
  ! 試行ごとに独立な乱数生成器 (MINSTD 法). 16807*s は 64bit に収まる.
  ! s を更新し, [0,N) の整数を返す.
  function draw(s, N) result(k)
    integer(8), intent(inout) :: s
    integer, intent(in) :: N
    integer :: k
    s = modulo(16807_8 * s, 2147483647_8)
    k = int(modulo(s, int(N, 8)))
  end function draw

  ! 1試行: N 種類が等確率で出るとき, 全種類そろうまでに引いた回数.
  ! collected を 64bit 整数のビットマスクで管理 (N <= 62 を想定).
  function one_trial(N, seed) result(draws)
    integer, intent(in) :: N
    integer(8), intent(in) :: seed
    integer(8) :: draws, s, got, full
    integer :: k
    s = modulo(seed, 2147483647_8)
    if (s <= 0) s = s + 2147483646_8
    got = 0_8
    full = ishft(1_8, N) - 1_8
    draws = 0_8
    do while (got /= full)
       k = draw(s, N)
       got = ior(got, ishft(1_8, k))
       draws = draws + 1_8
    end do
  end function one_trial
end module gacha_mod

program gacha
  use gacha_mod
  character(len=32) :: arg
  integer :: N, k
  integer(8) :: T, t_
  real(8) :: sum, sumsq, mean, var, H
  N = 10
  T = 1000000_8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) N
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) T
  end if
  sum = 0.0d0; sumsq = 0.0d0

  ! T 回の試行は互いに独立. 各試行の引き回数を集計する.
  ! TODO: 各試行は独立なので !$omp parallel do reduction(+:sum,sumsq) で並列化・集計せよ.
  do t_ = 1, T
     block
       integer(8) :: d
       d = one_trial(N, t_ * 2654435761_8 + 12345_8)
       sum = sum + real(d, 8)
       sumsq = sumsq + real(d, 8) * real(d, 8)
     end block
  end do
  ! TODO: 上の parallel do を閉じる !$omp end parallel do を書け.

  mean = sum / T
  var  = sumsq / T - mean * mean
  ! 理論期待値 = N * H_N
  H = 0.0d0
  do k = 1, N
     H = H + 1.0d0 / k
  end do
  print "(a,i0,a,i0,a,f0.3,a,f0.3,a,f0.3)", &
       "N=", N, ", trials=", T, ": 平均 ", mean, " 回 (理論 ", N * H, "), 標準偏差 ", sqrt(var)
end program gacha

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore gacha.f90 -o gacha_f90.exe

## 3-2. 実行
- ログインノードでそのまま実行 (数秒で終わるジョブ)

In [ ]:
%%bash_
./gacha_f90.exe

- Aquariusに投入

In [ ]:
%%bash_submit_a
./gacha_f90.exe

- 上記は以下と同値
- キューや制限時間などを変更したいときは適宜変更・追加

In [ ]:
%%bash_submit

#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./gacha_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:gacha.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:gacha.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:gacha.cpp}